In [1]:
pip install opencv-python mediapipe

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sounddevice-0.5.5-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
    --------------------------------------- 0.3/10.9 MB ? eta -:--:--
   ------ --------------------------------- 1.8/10.9 MB 6.3 MB/s eta 0:00:02
   ---------------- ----------------------- 4.5/10.9 MB 9.2 MB/s eta 0:00:01
   ------------------------ --------------- 6.8/10.9 MB 10.0 MB/s eta 0:00:01
   ---------------------------------- ----

In [1]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
urllib.request.urlretrieve(url, "face_landmarker.task")

('face_landmarker.task', <http.client.HTTPMessage at 0x2360c086bc0>)

In [15]:
import cv2
import os
import time
import random
import shutil
import urllib.request
from pathlib import Path
from datetime import datetime

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ==================================================
# CONFIG
# ==================================================

MODEL_DIR = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "face_landmarker.task")

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

DATASET_DIR = "emotion_dataset"
SPLIT_DIR = "emotion_dataset_split"

IMAGE_SIZE = 224

In [18]:
# ==================================================
# IP WEBCAM CONFIG
# ==================================================
# Sửa IP này theo địa chỉ hiện trên app IP Webcam của điện thoại bạn.
# Ví dụ điện thoại hiện: http://192.168.1.5:8080
# Thì dùng: http://192.168.1.5:8080/video

USE_PHONE_CAMERA = True
PHONE_CAMERA_URL = "https://192.168.0.101:8080/video"

# Nếu muốn quay lại webcam laptop thì đổi USE_PHONE_CAMERA = False
CAMERA_ID = 0

# Nếu camera trước bị ngược như selfie, để True
FLIP_FRAME = True

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_SEED = 42

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

CAPTURE_COOLDOWN_SECONDS = 0.8

emotions = {
    "anger": "tuc_gian",
    "contempt": "khinh_thuong",
    "disgust": "ghe_tom",
    "fear": "so_hai",
    "happy": "vui_ve",
    "neutral": "trung_tinh",
    "sad": "buon",
    "surprise": "ngac_nhien"
}

In [3]:
# ==================================================
# DOWNLOAD MODEL
# ==================================================

def download_model():
    os.makedirs(MODEL_DIR, exist_ok=True)

    if not os.path.exists(MODEL_PATH):
        print("Đang tải face_landmarker.task...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
        print(f"Đã tải xong model: {MODEL_PATH}")
    else:
        print(f"Model đã tồn tại: {MODEL_PATH}")


# ==================================================
# CREATE DATASET FOLDERS
# ==================================================

def create_dataset_folders():
    os.makedirs(DATASET_DIR, exist_ok=True)

    for emotion in emotions.keys():
        os.makedirs(os.path.join(DATASET_DIR, emotion), exist_ok=True)


In [4]:
# ==================================================
# UTILS
# ==================================================

last_timestamp_ms = 0


def get_timestamp_ms():
    global last_timestamp_ms

    timestamp_ms = int(time.monotonic() * 1000)

    if timestamp_ms <= last_timestamp_ms:
        timestamp_ms = last_timestamp_ms + 1

    last_timestamp_ms = timestamp_ms

    return timestamp_ms


def clamp(value, min_value, max_value):
    return max(min_value, min(value, max_value))


def draw_text(
    frame,
    text,
    position,
    color=(0, 255, 0),
    scale=0.7,
    thickness=2
):
    cv2.putText(
        frame,
        text,
        position,
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        color,
        thickness,
        cv2.LINE_AA
    )

In [5]:
# ==================================================
# INIT FACE LANDMARKER
# ==================================================

def create_landmarker():
    base_options = python.BaseOptions(
        model_asset_path=MODEL_PATH
    )

    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.6,
        min_face_presence_confidence=0.6,
        min_tracking_confidence=0.6,
        output_face_blendshapes=True
    )

    return vision.FaceLandmarker.create_from_options(options)

In [6]:
# ==================================================
# FACE CROP FUNCTION
# ==================================================

def get_face_crop(frame, landmarker):
    """
    Crop khuôn mặt:
    - Lấy từ cằm hất lên
    - Không lấy cổ
    - Lấy trán cao hơn một chút
    - Mở rộng nhẹ hai bên mặt
    """

    img_h, img_w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp_ms = get_timestamp_ms()

    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if not result.face_landmarks:
        return None, None, None

    face_landmarks = result.face_landmarks[0]

    xs = []
    ys = []

    for landmark in face_landmarks:
        x = int(landmark.x * img_w)
        y = int(landmark.y * img_h)

        xs.append(x)
        ys.append(y)

    if len(xs) == 0 or len(ys) == 0:
        return None, None, None

    x_min = clamp(min(xs), 0, img_w - 1)
    y_min = clamp(min(ys), 0, img_h - 1)
    x_max = clamp(max(xs), 0, img_w - 1)
    y_max = clamp(max(ys), 0, img_h - 1)

    w = x_max - x_min
    h = y_max - y_min

    if w <= 0 or h <= 0:
        return None, None, None
    
        # ==================================================
    # CROP THEO YÊU CẦU
    # ==================================================
    # x1/x2: rộng nhẹ hai bên mặt
    # y1: kéo lên để lấy trán cao hơn một chút
    # y2: chỉ tới cằm, không lấy cổ
    # ==================================================

    x1 = int(x_min - 0.08 * w)
    y1 = int(y_min - 0.12 * h)
    x2 = int(x_max + 0.08 * w)
    y2 = int(y_max + 0.02 * h)

    x1 = clamp(x1, 0, img_w - 1)
    y1 = clamp(y1, 0, img_h - 1)
    x2 = clamp(x2, 1, img_w)
    y2 = clamp(y2, 1, img_h)

    if x2 <= x1 or y2 <= y1:
        return None, None, None

    crop = frame[y1:y2, x1:x2]

    if crop.size == 0:
        return None, None, None

    top_blendshape = None

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        if len(blendshapes) > 0:
            top = max(blendshapes, key=lambda item: item.score)
            top_blendshape = f"{top.category_name}: {top.score:.2f}"

    return crop, (x1, y1, x2, y2), top_blendshape

In [7]:
# ==================================================
# SELECT EMOTION
# ==================================================

def select_emotion():
    print("\n===== DANH SÁCH CẢM XÚC =====")

    emotion_list = list(emotions.keys())

    for i, emotion in enumerate(emotion_list, start=1):
        print(f"{i}. {emotion} - {emotions[emotion]}")

    try:
        choice = int(input("\nNhập số thứ tự cảm xúc muốn chụp: "))
    except ValueError:
        print("Bạn phải nhập số.")
        return None

    if choice < 1 or choice > len(emotion_list):
        print("Lựa chọn không hợp lệ.")
        return None

    selected_emotion = emotion_list[choice - 1]

    print(
        f"\nBạn đã chọn: "
        f"{selected_emotion} - {emotions[selected_emotion]}"
    )

    return selected_emotion

In [8]:
# ==================================================
# SAVE IMAGE
# ==================================================

def save_face_image(crop, selected_emotion):
    save_image = cv2.resize(
        crop,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=cv2.INTER_AREA
    )

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

    filename = f"{selected_emotion}_{timestamp}.jpg"

    save_path = os.path.join(
        DATASET_DIR,
        selected_emotion,
        filename
    )

    success = cv2.imwrite(save_path, save_image)

    return success, save_path


# ==================================================
# OPEN CAMERA
# ==================================================

def open_camera():
    if USE_PHONE_CAMERA:
        print(f"Đang kết nối IP Webcam: {PHONE_CAMERA_URL}")
        cap = cv2.VideoCapture(PHONE_CAMERA_URL)
    else:
        print(f"Đang mở webcam laptop ID: {CAMERA_ID}")
        cap = cv2.VideoCapture(CAMERA_ID)

    return cap


In [9]:
# ==================================================
# COLLECT DATASET BY SPACE
# ==================================================

def collect_dataset():
    download_model()
    create_dataset_folders()

    landmarker = create_landmarker()

    selected_emotion = select_emotion()

    if selected_emotion is None:
        landmarker.close()
        return

    cap = open_camera()

    if not cap.isOpened():
        print("\nKhông mở được camera.")
        print("Kiểm tra lại:")
        print("1. Điện thoại và laptop cùng Wi-Fi")
        print("2. App IP Webcam đã bấm Start server")
        print("3. PHONE_CAMERA_URL đúng IP, ví dụ http://192.168.1.5:8080/video")
        print("4. Camera trước đã được chọn trong app IP Webcam")
        landmarker.close()
        return

    # Có thể không tác dụng với IP Webcam, nhưng vẫn giữ cho webcam laptop
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    print("\n===== HƯỚNG DẪN =====")
    print("SPACE : Chụp và lưu ảnh")
    print("Q/ESC : Thoát và tự chia train/val/test")
    print("=====================\n")

    count = 0
    last_capture_time = 0.0

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Không đọc được hình ảnh từ camera.")
            print("Nếu dùng IP Webcam, hãy kiểm tra lại URL hoặc mở lại Start server.")
            break

        if FLIP_FRAME:
            frame = cv2.flip(frame, 1)

        display_frame = frame.copy()

        crop, crop_box, top_blendshape = get_face_crop(
            frame,
            landmarker
        )

        if crop_box is not None:
            x1, y1, x2, y2 = crop_box

            cv2.rectangle(
                display_frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            status_text = "Face detected - Press SPACE to save"
            status_color = (0, 255, 0)

        else:
            status_text = "No face detected"
            status_color = (0, 0, 255)

        draw_text(
            display_frame,
            f"Emotion: {selected_emotion}",
            (20, 40),
            (0, 255, 0),
            0.9,
            2
        )

        draw_text(
            display_frame,
            status_text,
            (20, 80),
            status_color,
            0.7,
            2
        )

        draw_text(
            display_frame,
            f"Captured: {count}",
            (20, 120),
            (255, 255, 0),
            0.7,
            2
        )

        draw_text(
            display_frame,
            "SPACE: capture | Q/ESC: quit and split",
            (20, 160),
            (0, 255, 255),
            0.7,
            2
        )

        if top_blendshape is not None:
            draw_text(
                display_frame,
                f"Blendshape: {top_blendshape}",
                (20, 200),
                (255, 255, 255),
                0.6,
                2
            )

        cv2.imshow(
            "Emotion Dataset Collector - IP Webcam",
            display_frame
        )

        key = cv2.waitKey(1) & 0xFF

        # SPACE để lưu ảnh
        if key == ord(" "):
            now = time.time()

            if now - last_capture_time < CAPTURE_COOLDOWN_SECONDS:
                continue

            if crop is None:
                print("Không phát hiện khuôn mặt, chưa lưu ảnh.")
                continue

            success, save_path = save_face_image(
                crop,
                selected_emotion
            )

            if success:
                count += 1
                last_capture_time = now
                print(f"[{count}] Đã lưu: {save_path}")
            else:
                print("Không lưu được ảnh.")

        # Q hoặc ESC để thoát
        elif key == ord("q") or key == ord("Q") or key == 27:
            break

    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()

    print(
        f"\nHoàn thành. Đã chụp {count} ảnh "
        f"cho cảm xúc {selected_emotion}."
    )

In [13]:
# ==================================================
# SPLIT DATASET 70:15:15
# ==================================================

def split_dataset():
    total_ratio = TRAIN_RATIO + VAL_RATIO + TEST_RATIO

    if abs(total_ratio - 1.0) > 1e-6:
        raise ValueError(
            "Tổng TRAIN_RATIO + VAL_RATIO + TEST_RATIO phải bằng 1.0"
        )

    random.seed(RANDOM_SEED)

    source_path = Path(DATASET_DIR)
    output_path = Path(SPLIT_DIR)

    if not source_path.exists():
        print(f"Không tìm thấy thư mục nguồn: {DATASET_DIR}")
        return

    if output_path.exists():
        print(f"Đang xóa thư mục chia cũ: {SPLIT_DIR}")
        shutil.rmtree(output_path)

    for split in ["train", "val", "test"]:
        (output_path / split).mkdir(parents=True, exist_ok=True)

    class_dirs = [
        d for d in source_path.iterdir()
        if d.is_dir()
    ]

    if len(class_dirs) == 0:
        print("Không tìm thấy class nào trong emotion_dataset.")
        return

    summary = {}

    for class_dir in class_dirs:
        class_name = class_dir.name

        images = [
            file for file in class_dir.iterdir()
            if file.is_file()
            and file.suffix.lower() in IMAGE_EXTENSIONS
        ]

        random.shuffle(images)

        total_images = len(images)

        if total_images == 0:
            print(f"Bỏ qua class '{class_name}' vì không có ảnh.")
            continue

        train_count = int(total_images * TRAIN_RATIO)
        val_count = int(total_images * VAL_RATIO)

        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]

        split_data = {
            "train": train_images,
            "val": val_images,
            "test": test_images
        }

        for split_name, split_images in split_data.items():
            split_class_dir = output_path / split_name / class_name
            split_class_dir.mkdir(parents=True, exist_ok=True)

            for img_path in split_images:
                dst_path = split_class_dir / img_path.name
                shutil.copy2(img_path, dst_path)

        summary[class_name] = {
            "total": total_images,
            "train": len(train_images),
            "val": len(val_images),
            "test": len(test_images)
        }

    print("\n===== KẾT QUẢ CHIA DATASET =====")

    total_all = 0
    total_train = 0
    total_val = 0
    total_test = 0

    for class_name, info in summary.items():
        print(
            f"{class_name:15s} | "
            f"Total: {info['total']:5d} | "
            f"Train: {info['train']:5d} | "
            f"Val: {info['val']:5d} | "
            f"Test: {info['test']:5d}"
        )

        total_all += info["total"]
        total_train += info["train"]
        total_val += info["val"]
        total_test += info["test"]

    print("-----------------------------------------------")
    print(
        f"{'ALL':15s} | "
        f"Total: {total_all:5d} | "
        f"Train: {total_train:5d} | "
        f"Val: {total_val:5d} | "
        f"Test: {total_test:5d}"
    )

    print("\nHoàn thành chia dataset.")
    print(f"Dataset đã chia nằm tại: {SPLIT_DIR}")

In [ ]:
# ==================================================
# MAIN
# ==================================================

if __name__ == "__main__":
    collect_dataset()

    print("\nĐang tự động chia dataset thành train/val/test = 70/15/15...")
    split_dataset()


Model đã tồn tại: models\face_landmarker.task

===== DANH SÁCH CẢM XÚC =====
1. anger - tuc_gian
2. contempt - khinh_thuong
3. disgust - ghe_tom
4. fear - so_hai
5. happy - vui_ve
6. neutral - trung_tinh
7. sad - buon
8. surprise - ngac_nhien
Bạn phải nhập số.

Đang tự động chia dataset thành train/val/test = 70/15/15...
Bỏ qua class 'anger' vì không có ảnh.
Bỏ qua class 'contempt' vì không có ảnh.
Bỏ qua class 'disgust' vì không có ảnh.
Bỏ qua class 'fear' vì không có ảnh.
Bỏ qua class 'happy' vì không có ảnh.
Bỏ qua class 'neutral' vì không có ảnh.
Bỏ qua class 'sad' vì không có ảnh.
Bỏ qua class 'surprise' vì không có ảnh.

===== KẾT QUẢ CHIA DATASET =====
-----------------------------------------------
ALL             | Total:     0 | Train:     0 | Val:     0 | Test:     0

Hoàn thành chia dataset.
Dataset đã chia nằm tại: emotion_dataset_split


: 